# 08 — Full Simulation Loop (Issue 10)

End-to-end simulation runner that wires all components together:
1. Configure simulation parameters via `SimulationConfig`
2. Run the daily loop: phase ordering → end-of-day survey → memory management
3. Collect results into `SimulationResults` (opinion trajectories, reflections, baseline)
4. Save CSV output and produce panel plots of opinion trajectories

**Covers:** Issue 10 (Simulation Loop + Phase Ordering + Output)  
**Depends on:** All previous issues (blocking: Issues 8 and 9)

In [ ]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key
# from cag.abm.simulation import SimulationConfig, run_simulation
# from cag.io.output import save_results, plot_opinion_trajectories

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)
import pandas as pd
import matplotlib.pyplot as plt

print("Imports OK")

## 1. Configure Simulation

In [ ]:
# TODO: Create SimulationConfig with small demo parameters
# config = SimulationConfig(
#     n_citizens=10,        # small demo
#     n_days=3,             # smoke test
#     target_policies=[ClimatePolicyID.CARBON_TAX],
#     phase_order=["P-A", "P-B", "C"],
#     max_shift=1,
#     k_conversations_per_day=3,
#     llm_model="gpt-4o-mini",
#     random_seed=42,
# )
# print(config)

## 2. Load Data & Build Environment

In [ ]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

api_key = load_api_key("../data/api_key.csv")

print(f"Total citizens: {len(sn.agents_active)}")

## 3. Run Simulation (Smoke Test)

Run a 3-day simulation with 10 agents to validate the full pipeline.

In [ ]:
# TODO: Run simulation
# results = run_simulation(config, sn)
# print(f"Simulation complete: {config.n_days} days, {len(sn.agents_active)} agents")

## 4. Opinion Trajectories

In [ ]:
# TODO: Display opinion_trajectories DataFrame
# results.opinion_trajectories.head(20)

## 5. Panel Plot — Opinion Over Time

X-axis: simulation day, Y-axis: opinion score (-3 to +3).  
Individual agent lines (alpha ~0.2) with bold group-mean overlaid.

In [ ]:
# TODO: Plot opinion trajectories
# plot_opinion_trajectories(results, output_dir="../data/output/experiments")

## 6. Phase Ordering Comparison

Run with two different phase orderings to verify they produce different trajectories.

In [ ]:
# TODO: Compare phase orderings
# config_alt = SimulationConfig(
#     n_citizens=10, n_days=3,
#     phase_order=["C", "P-B", "P-A"],  # reversed
#     random_seed=42,
# )
# results_alt = run_simulation(config_alt, sn_alt)
# assert not results.opinion_trajectories.equals(results_alt.opinion_trajectories)

## 7. Save Results

In [ ]:
# TODO: Save CSV output and config.json
# save_results(results, output_dir="../data/output/experiments")
# print("Results saved")

## 8. Sanity Checks

In [ ]:
# TODO: Sanity checks
# - opinion_trajectories has correct schema
# - n_days * n_agents rows in trajectory DataFrame
# - No clamped shift exceeds max_shift
# - CSV files exist in timestamped directory
# - config.json is valid and matches SimulationConfig
# - Panel plot PNG exists

## 9. LLM Call Summary

In [ ]:
# TODO: Total LLM calls for the full simulation
# Per day: generate_message(2) + broadcast_reflections(n_a + n_b) + peer_messages(n) + peer_reflections(n) + survey(n)
# Per simulation: per_day * n_days + memory_compression_calls